# ITMO NLP HW1: Topic Classification (Lenta.ru)

Цель: классификация новостей по `topic` на репрезентативной выборке из 100_000 текстов.


## 0. Формат решения

- Основные вычисления вынесены в `src/hw1_pipeline.py`.
- Ноутбук используется как воспроизводимый отчёт по уже посчитанным артефактам.
- Основной финальный прогон для сдачи: `artifacts/run_v2`.
- Дополнительный прогон для сравнения: `artifacts/run_v8_fast`.


In [4]:
import json
from pathlib import Path
import pandas as pd

SEED = 42
PRIMARY_RUN = "run_v2"
SECONDARY_RUN = "run_v8_fast"

primary_dir = Path("artifacts") / PRIMARY_RUN
secondary_dir = Path("artifacts") / SECONDARY_RUN
metrics_dir = primary_dir / "metrics"
reports_dir = primary_dir / "reports"


## 1. Загрузка подтверждённых результатов

Ниже ноутбук читает артефакты основного прогона `run_v2`, который считаем финальным по `macro-F1`.


In [5]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

split_stats = load_json(metrics_dir / "split_stats.json")
baseline = load_json(metrics_dir / "dummy_baseline.json")
base_models = load_json(metrics_dir / "base_models_val.json")
tuning = load_json(metrics_dir / "tuning_results.json")
final_test = load_json(metrics_dir / "final_test_metrics.json")
secondary_test = load_json(secondary_dir / "metrics" / "final_test_metrics.json")

summary_df = pd.DataFrame([
    {"stage": "dummy_most_frequent", "accuracy": baseline["most_frequent"]["accuracy"], "macro_f1": baseline["most_frequent"]["macro_f1"]},
    {"stage": "count_logreg_val", "accuracy": base_models["count_logreg"]["accuracy"], "macro_f1": base_models["count_logreg"]["macro_f1"]},
    {"stage": "tfidf_logreg_val", "accuracy": base_models["tfidf_logreg"]["accuracy"], "macro_f1": base_models["tfidf_logreg"]["macro_f1"]},
    {"stage": "run_v2_test", "accuracy": final_test["accuracy"], "macro_f1": final_test["macro_f1"]},
    {"stage": "run_v8_fast_test", "accuracy": secondary_test["accuracy"], "macro_f1": secondary_test["macro_f1"]},
]).round(4)

summary_df


,stage,accuracy,macro_f1
0,dummy_most_frequent,0.2171,0.0188
1,count_logreg_val,0.8179,0.6177
2,tfidf_logreg_val,0.8066,0.5254
3,run_v2_test,0.8135,0.7040
4,run_v8_fast_test,0.8186,0.6933


## 2. Ключевые параметры и итоговые метрики


In [6]:
pd.DataFrame([
    {"parameter": "sample_size", "value": 100_000},
    {"parameter": "split", "value": "60/20/20 stratified"},
    {"parameter": "num_classes", "value": split_stats["num_classes"]},
    {"parameter": "best_cv_macro_f1", "value": round(tuning["best_score_cv_macro_f1"], 4)},
    {"parameter": "final_model", "value": PRIMARY_RUN},
    {"parameter": "final_test_accuracy", "value": round(final_test["accuracy"], 4)},
    {"parameter": "final_test_macro_f1", "value": round(final_test["macro_f1"], 4)},
])


,parameter,value
0,sample_size,100000
1,split,60/20/20 stratified
2,num_classes,19
3,best_cv_macro_f1,0.5652
4,final_model,run_v2
5,final_test_accuracy,0.8135
6,final_test_macro_f1,0.704


## 3. Анализ ошибок лучшего прогона


In [7]:
misclassified_path = reports_dir / "misclassified_examples.csv"
error_report_path = reports_dir / "error_analysis.md"

misclassified = pd.read_csv(misclassified_path)
misclassified.head(8)


,true_label,pred_label,text_preview
0,Россия,Бывший СССР,в бишкекском цирке поймали россиянина с героин...
1,Интернет и СМИ,Мир,журналистов попросили не приезжать на годовщин...
2,Из жизни,Россия,в кемерове женщина ушибла поясницу при падении...
3,Из жизни,Культура,самые уродливые восковые фигуры великобритании...
4,Бывший СССР,Из жизни,полиция таллина закрыла дело о дважды упавшей ...
5,Спорт,Из жизни,чешский нападающий нхл проиграл виртуальному к...
6,Из жизни,Мир,охрана британской короны превратилась в каторг...
7,Мир,Дом,у бориса беккера отобрали дом в испании испанс...


In [8]:
print(error_report_path.read_text(encoding="utf-8"))


# Error analysis

Total test errors: 3730

## Top confusion pairs

- `Россия` -> `Мир`: 315
- `Мир` -> `Россия`: 209
- `Мир` -> `Из жизни`: 174
- `Россия` -> `Силовые структуры`: 169
- `Россия` -> `Бывший СССР`: 107
- `Россия` -> `Интернет и СМИ`: 92
- `Россия` -> `Экономика`: 92
- `Мир` -> `Бывший СССР`: 88
- `Экономика` -> `Россия`: 75
- `Россия` -> `Из жизни`: 70


## 4. Выводы

- Использован репрезентативный стратифицированный сэмпл на 100_000 записей из `lenta-ru-news`.
- Лёгкая предобработка (`title + text`, lowercase, очистка спецсимволов, нормализация пробелов) дала приемлемый баланс качества и скорости.
- Dummy-бейзлайн очень низкий (`accuracy=0.217`, `macro-F1=0.019`), что подтверждает нетривиальность задачи.
- На валидации `CountVectorizer + LogisticRegression` оказался лучше `TfidfVectorizer + LogisticRegression`:
  - Count: `accuracy=0.8179`, `macro-F1=0.6177`
  - TF-IDF: `accuracy=0.8067`, `macro-F1=0.5254`
- После CV-тюнинга (RandomizedSearchCV) итог на test (прогон `run_v2`):
  - `accuracy=0.8135`
  - `macro-F1=0.7040`
- Дополнительный тюнинг (прогон `run_v8_fast`) дал:
  - `accuracy=0.8186`
  - `macro-F1=0.6933`
- Если приоритет — `macro-F1` (основная метрика при дисбалансе классов), то лучшим остаётся `run_v2`; если приоритет — только accuracy, то `run_v8_fast` немного лучше.
- Наиболее частые ошибки: путаница между темами `Россия` и `Мир`, а также пересечения с `Бывший СССР` и `Силовые структуры`.
- Практический вывод: текущий пайплайн существенно превосходит baseline и подходит как рабочее решение; для роста `macro-F1` стоит расширить targeted-тюнинг вокруг параметров `run_v2` и добавить более аккуратную работу с редкими классами.
